In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline
)
from sklearn.metrics import accuracy_score
import numpy as np

In [ ]:
dataset = load_dataset("imdb")

train_data = dataset["train"].shuffle(seed=42).select(range(1000))
test_data = dataset["test"].shuffle(seed=42).select(range(100))

train_split = train_data.select(range(900))
val_split = train_data.select(range(900, 1000))

model_checkpoint = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True)

train_tokenized = train_split.map(tokenize_function)
val_tokenized = val_split.map(tokenize_function)
test_tokenized = test_data.map(tokenize_function)

print(tokenizer.convert_ids_to_tokens(train_tokenized[0]["input_ids"]))

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(arg):
    predictions, labels = arg
    acc = accuracy_score(labels, np.argmax(predictions, axis=1))
    return {
        "accuracy": acc
    }

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
results = trainer.evaluate(test_tokenized)
print(results)

sentiment_pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
print(sentiment_pipe("This movie was a masterpiece!"))
print(sentiment_pipe("Absolutely terrible. Never again."))

# Baseline
# {'eval_loss': 0.3171509802341461, 'eval_accuracy': 0.87, 'eval_runtime': 0.5494, 'eval_samples_per_second': 182.026, 'eval_steps_per_second': 23.663, 'epoch': 1.0}
# [{'label': 'POSITIVE', 'score': 0.9271114468574524}]
# [{'label': 'NEGATIVE', 'score': 0.9444614052772522}]

# FacebookAI/roberta-base
# {'eval_loss': 0.23342078924179077, 'eval_accuracy': 0.91, 'eval_runtime': 0.5208, 'eval_samples_per_second': 192.004, 'eval_steps_per_second': 24.961, 'epoch': 1.0}
# [{'label': 'POSITIVE', 'score': 0.9452106952667236}]
# [{'label': 'NEGATIVE', 'score': 0.9697701930999756}]

# distilbert/distilbert-base-uncased
# {'eval_loss': 0.41859713196754456, 'eval_accuracy': 0.85, 'eval_runtime': 0.2805, 'eval_samples_per_second': 356.468, 'eval_steps_per_second': 46.341, 'epoch': 1.0}
# [{'label': 'POSITIVE', 'score': 0.9434113502502441}]
# [{'label': 'NEGATIVE', 'score': 0.9497330188751221}]